In [5]:
from scipy.stats import norm
import numpy as np
import time
import import_ipynb
from GBM import simulate_GBM, simulate_GBM_antithetic
from european import european_mc, european_mc_antithetic
import matplotlib.pyplot as plt
from scipy.stats import qmc

In [1]:
def asian_mc(S0, K, r, sigma, T, n, n_paths, average_type="arithmetic", type="call"):
    t, S = simulate_GBM(S0, r, sigma, T, n, n_paths)
    
    if average_type == "arithmetic":
        avg = S[:, 1:].mean(axis=1)  # exclude t=0 column, average over the path
    else:  # geometric
        avg = np.exp(np.log(S[:, 1:]).mean(axis=1))
    
    if type == "call":
        payoff = np.maximum(avg - K, 0)
    else:
        payoff = np.maximum(K - avg, 0)
    
    discounted = np.exp(-r*T) * payoff
    price = discounted.mean()
    se = discounted.std(ddof=1) / np.sqrt(n_paths)
    return price, se

In [2]:
def geometric_asian_bs(S0, K, r, sigma, T, type="call"):
    sigma_adj = sigma / np.sqrt(3)
    r_adj = 0.5 * (r - 0.5*sigma**2 + sigma_adj**2)
    
    d1 = (np.log(S0/K) + (r_adj + 0.5*sigma_adj**2)*T) / (sigma_adj*np.sqrt(T))
    d2 = d1 - sigma_adj*np.sqrt(T)
    
    if type == "call":
        price = np.exp(-r*T) * (S0*np.exp(r_adj*T)*norm.cdf(d1) - K*norm.cdf(d2))
    else:
        price = np.exp(-r*T) * (K*norm.cdf(-d2) - S0*np.exp(r_adj*T)*norm.cdf(-d1))
    return price

In [3]:
def price_lookback_mc(S0, r, sigma, T, n, n_paths, type="call"):
    t, S = simulate_GBM(S0, r, sigma, T, n, n_paths)
    S_T = S[:, -1]
    
    if type == "call":
        running_min = S.min(axis=1)
        payoff = S_T - running_min
    else:
        running_max = S.max(axis=1)
        payoff = running_max - S_T
    
    discounted = np.exp(-r*T) * payoff
    price = discounted.mean()
    se = discounted.std(ddof=1) / np.sqrt(n_paths)
    return price, se

In [4]:
def barrier_mc(S0, K, B, r, sigma, T, n, n_paths, type="call", barrier_type="up-and-out"):
    t, S = simulate_GBM(S0, r, sigma, T, n, n_paths)
    S_T = S[:, -1]
    
    if type == "call":
        payoff = np.maximum(S_T - K, 0)
    else:
        payoff = np.maximum(K - S_T, 0)
    
    if barrier_type == "up-and-out":
        breached = (S.max(axis=1) >= B)
        payoff = np.where(breached, 0, payoff)
    elif barrier_type == "down-and-out":
        breached = (S.min(axis=1) <= B)
        payoff = np.where(breached, 0, payoff)
    elif barrier_type == "up-and-in":
        breached = (S.max(axis=1) >= B)
        payoff = np.where(breached, payoff, 0)
    elif barrier_type == "down-and-in":
        breached = (S.min(axis=1) <= B)
        payoff = np.where(breached, payoff, 0)
    else:
        raise ValueError("Unknown barrier_type")
    
    discounted = np.exp(-r*T) * payoff
    price = discounted.mean()
    se = discounted.std(ddof=1) / np.sqrt(n_paths)
    return price, se